# Operations Resolver Agent — Crew Demo Notebook

Quest #4, Part 2 (Place-IL). This notebook is a runnable walkthrough of the
**Researcher → Decision → Comms** crew described in [`README.md`](README.md)'s
"Part 2 — Distributed Agent Crew" section — it does not repeat that design
writeup (the flow diagram, the authority-separation rationale, the full
guardrails table), only demonstrates the crew working, live, against the
real Anthropic API and the real (unmodified) starter-kit tools.

Every cell that calls `crew.handle_ticket(...)` makes three real model calls
(one per agent) — no mocking, no fake model. Re-running this notebook
re-spends live API calls. A couple of cells are deliberately deterministic
(no API call) — they demonstrate a guardrail directly, the same way
`demo.ipynb`'s security-review cells do.

**What this notebook shows, in order:**

1. A clean case — eligible policy, low fraud risk, no escalation, no alert
2. The headline fraud trap — `ORD-1005`, where `check_return_policy` alone
   says `ELIGIBLE` but the fraud engine overrides it
3. Authority separation is physical, not just prompted — inspecting each
   agent's real tool registry
4. The crew's own regression suite (`scripts/run_crew_scenarios.py`'s
   scenarios), run live and summarized as a table
5. A guardrail spotlight — catching a Comms leak deterministically before
   it can reach the customer

A real, saved transcript of this same regression suite already lives in
[`docs/evidence/`](docs/evidence/) (stdout + structured JSON logs, 6/6
scenarios matched) — this notebook is the interactive companion to that,
not a replacement for it.


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd() / "scripts"))

from resolver_agent.crew import OperationsCrew
from resolver_agent.crew.schemas import CrewResult
from resolver_agent.logging_utils import configure_logging

import multi_agent_tools as mat  # starter-kit/ is already on sys.path via resolver_agent.agent

configure_logging()  # default WARNING -- a clean run prints nothing to stderr

crew = OperationsCrew()


def show_crew(result: CrewResult, title: str = "") -> None:
    if title:
        print(f"=== {title} ===")
    print(f"order_id           : {result.order_id}")
    decision = result.decision
    if decision is not None:
        print(f"refund_status      : {decision.refund_status}  (policy verdict: {decision.verdict})")
        rr = decision.risk_report
        print(f"fraud risk_band    : {rr.risk_band}  (score {rr.risk_score}/100)")
        print(f"requested/approved : {decision.requested_amount} / {decision.approved_amount}")
    else:
        print(f"decision           : None  (pipeline stopped early)")
    print(f"stopped_reason     : {result.stopped_reason}")
    print(f"alert_sent         : {result.alert_sent}")
    if result.escalation:
        print(
            f"escalation route   : channel={result.escalation.get('channel')} "
            f"required={result.escalation.get('escalation_required')}"
        )
    print()
    print("reasoning_chain:")
    for line in result.reasoning_chain:
        print(f"  - {line}")
    print()
    print("customer_response:")
    print(f"  {result.customer_response}")


## 1. Clean case — eligible policy, low fraud risk, no escalation, no alert

`ORD-1001`: earbuds arrived cracked. Expect the Researcher to clear the
customer, the Decision agent's `check_return_policy` + `process_refund` to
approve it outright, and Comms to send neither an escalation nor a Slack
alert — the same clean-case baseline `run_crew_scenarios.py` calls `B3`.


In [2]:
outbox_before = len(mat.read_outbox())

result_clean = crew.handle_ticket(
    "Hi, I'm Maya. My earbuds from order ORD-1001 arrived cracked right out "
    "of the box. Can you sort this out?"
)
show_crew(result_clean, "Scenario 1 -- clean case")

outbox_after = len(mat.read_outbox())
print(f"\noutbox entries: {outbox_before} -> {outbox_after} (should be unchanged)")


=== Scenario 1 -- clean case ===
order_id           : ORD-1001
refund_status      : APPROVED  (policy verdict: ELIGIBLE)
fraud risk_band    : low  (score 0/100)
requested/approved : 35.0 / 35.0
stopped_reason     : stop
alert_sent         : False
escalation route   : channel=None required=False

reasoning_chain:
  - Risk report: order ORD-1001, user USR-101, risk_band=low (score 0), blocks_automatic_refund=False.
  - Decision: verdict=ELIGIBLE, refund_status=APPROVED, requested_amount=35.0, approved_amount=35.0.
  - Escalation route: escalation_required=False, channel=None.
  - Alert sent: False.

customer_response:
  Thank you for reporting the damage to your order. We've reviewed your claim and confirmed that you're eligible for a refund under our return policy. Your refund of $35.00 has been approved and processed under refund ID RF-1001-3500. You should see the credit back to your original payment method within 3–5 business days. We appreciate your business and apologize for the in

## 2. The headline fraud trap — `ORD-1005`

This is the brief's own central trap: the claim is **completely legitimate**
on paper — `check_return_policy` returns `ELIGIBLE` — but `audit_fraud_risk`
scores this same order/customer 90/100, `risk_band="high"`. A team that only
reads the policy verdict pays out a fraud-flagged claim. The Decision agent
sees the *full* `RiskReport` (not just a `"high"` label) via the Researcher →
Decision handoff, and is expected to override the policy verdict to
`ESCALATION_REQUIRED`; Comms is then expected to route a Slack alert with
the real risk detail as payload — while telling the customer something
honest that never mentions the fraud flag by name.


In [3]:
outbox_before = len(mat.read_outbox())

result_fraud = crew.handle_ticket(
    "This is Ronen, order ORD-1005. The tablet screen was smashed on "
    "arrival. Refund me the full 480 dollars, this keeps happening."
)
show_crew(result_fraud, "Scenario 2 -- ORD-1005 fraud override")

outbox_after = len(mat.read_outbox())
print(f"\noutbox entries: {outbox_before} -> {outbox_after}")

if result_fraud.alert_record:
    print("\nalert_record actually written to the outbox:")
    print(json.dumps(result_fraud.alert_record, indent=2))

leak_terms = ("fraud", "risk score", "flagged", "suspicious")
leaked = [t for t in leak_terms if t in result_fraud.customer_response.lower()]
print(
    f"\ncustomer_response mentions any of {leak_terms}: "
    f"{leaked or 'none -- the fraud flag stayed internal, as required'}"
)


=== Scenario 2 -- ORD-1005 fraud override ===
order_id           : ORD-1005
refund_status      : ESCALATION_REQUIRED  (policy verdict: ELIGIBLE)
fraud risk_band    : high  (score 90/100)
requested/approved : 480.0 / None
stopped_reason     : stop
alert_sent         : True
escalation route   : channel=#fraud-security required=True

reasoning_chain:
  - Risk report: order ORD-1005, user USR-105, risk_band=high (score 90), blocks_automatic_refund=True.
  - Decision: verdict=ELIGIBLE, refund_status=ESCALATION_REQUIRED, requested_amount=480.0, approved_amount=None.
  - Escalation route: escalation_required=True, channel=#fraud-security.
  - Alert sent: True.

customer_response:
  Thank you for submitting your refund request for order ORD-1005. We understand how frustrating it is to receive a damaged item.

Your request is currently under review as part of our standard verification process. Our team will investigate your claim and contact you within 1-2 business days with an update.

We appr

## 3. Authority separation is physical, not just prompted

Each agent's tool registry is built from a fixed tuple of tool names, one
per role, matching the starter kit's own `multi_agent_tools.TOOL_OWNERSHIP`
map (`README.md`'s "Authority separation is physical, not just prompted"
section). This cell doesn't call the model at all — it just inspects the
real `tool_registry` dict each agent was constructed with, proving
`process_refund` is a name that literally does not exist for two of the
three agents, not merely a name the prompt tells them not to use.


In [4]:
print("Researcher tools:", sorted(crew.researcher.tool_registry.keys()))
print("Decision tools   :", sorted(crew.decision_agent.tool_registry.keys()))
print("Comms tools      :", sorted(crew.comms_agent._base_tool_registry.keys()))

for name, agent in [("Researcher", crew.researcher), ("Comms", crew.comms_agent)]:
    registry = agent.tool_registry if hasattr(agent, "tool_registry") else agent._base_tool_registry
    assert "process_refund" not in registry, f"{name} should never be able to reach process_refund"

print("\nprocess_refund is unreachable for Researcher and Comms -- confirmed by construction, not by prompt.")
print(f"Only DecisionAgent's registry contains it: {'process_refund' in crew.decision_agent.tool_registry}")


Researcher tools: ['audit_fraud_risk', 'get_order_details', 'get_user_profile']
Decision tools   : ['check_return_policy', 'process_refund']
Comms tools      : ['get_escalation_route', 'send_slack_alert']

process_refund is unreachable for Researcher and Comms -- confirmed by construction, not by prompt.
Only DecisionAgent's registry contains it: True


## 4. Full scenario suite — the crew's own regression check, live

Same six scenarios as [`scripts/run_crew_scenarios.py`](scripts/run_crew_scenarios.py)
(the headline fraud trap, a second high-value fraud case, the clean baseline,
and three Part 1 regression spot-checks run through the full crew). This is
the crew's actual judgment against a live model across three agents per
ticket — it can vary run to run, unlike the deterministic
`starter-kit/examples/verify_scenarios.py` check. A real, saved transcript of
this exact suite is in [`docs/evidence/`](docs/evidence/) if you don't want
to spend your own API calls re-running it.


In [5]:
from run_crew_scenarios import SCENARIOS  # same list scripts/run_crew_scenarios.py runs

rows = []
for scenario in SCENARIOS:
    outbox_before = len(mat.read_outbox())
    result = crew.handle_ticket(scenario["ticket"])
    outbox_after = len(mat.read_outbox())
    alert_written = outbox_after > outbox_before

    status = result.decision.refund_status if result.decision else result.stopped_reason
    rows.append({
        "id": scenario["id"],
        "title": scenario["title"],
        "expected_status": scenario["expected_refund_status"],
        "actual_status": status,
        "status_match": status == scenario["expected_refund_status"],
        "expected_alert": scenario["expect_alert"],
        "alert_written": alert_written,
        "alert_match": alert_written == scenario["expect_alert"],
    })

df = pd.DataFrame(rows)
df["match"] = df["status_match"] & df["alert_match"]
print(f"{df['match'].sum()}/{len(df)} scenarios matched (status and alert both as expected).\n")
df


{"level": "WARNING", "logger": "resolver_agent.crew.decision.agent", "event": "decision.corrected", "correction_count": 2, "case_id": "faf6a4a7", "agent_role": "decision"}


6/6 scenarios matched (status and alert both as expected).



,id,title,expected_status,actual_status,status_match,expected_alert,alert_written,alert_match,match
0,B1,"The headline fraud case -- ELIGIBLE policy, hi...",ESCALATION_REQUIRED,ESCALATION_REQUIRED,True,True,True,True,True
1,B2,"New account, high value, item never arrived",ESCALATION_REQUIRED,ESCALATION_REQUIRED,True,True,True,True,True
2,B3,"Clean case -- no escalation, no alert",APPROVED,APPROVED,True,False,False,True,True
3,P1-2,Part 1 regression -- authority breach still es...,ESCALATION_REQUIRED,ESCALATION_REQUIRED,True,True,True,True,True
4,P1-3,Part 1 regression -- outside the return window...,REJECTED,REJECTED,True,True,True,True,True
5,P1-8,Part 1 regression -- non-returnable category s...,REJECTED,REJECTED,True,True,True,True,True


## 5. Guardrail spotlight — catching a Comms leak before it reaches the customer

Live testing (see `README.md`'s "Choosing a model per agent" section) found
Comms could repeat a stale, since-corrected refund figure straight out of
`decision.rationale`, or describe a refund as already approved on a case
that wasn't. A prompt instruction not to do this is not a guarantee — this
cell reproduces both leaks directly against
[`resolver_agent/crew/comms/output_tool.py`](resolver_agent/crew/comms/output_tool.py)'s
deterministic checks (no live model call needed), the same way `demo.ipynb`
demonstrates its own security-review guardrail directly against
`enforce_resolution()`.


In [6]:
from resolver_agent.crew.comms.output_tool import find_premature_approval_language, find_stale_refund_detail
from resolver_agent.crew.schemas import Decision, RiskReport

demo_risk_report = RiskReport(
    order_id="ORD-1002", user_id="USR-102", risk_score=10, risk_band="low",
    action_hint="proceed", triggered_rules=[], evidence={},
    blocks_automatic_refund=False, requires_security_channel=False,
    rulebook_version="demo-v1",
)
demo_decision = Decision(
    order_id="ORD-1002", user_id="USR-102", verdict="ELIGIBLE", eligible=True,
    refund_status="ESCALATION_REQUIRED",  # the real, current outcome
    requested_amount=150.0, approved_amount=None, refund_id=None,
    applicable_policies=["STANDARD"],
    rationale="An earlier process_refund attempt at the $50 cap returned RF-0001 before this case was escalated.",
    risk_report=demo_risk_report,
)

leaky_response = (
    "Good news -- your refund of $150.00 has been approved and refund "
    "RF-0001 is on its way."
)

stale_detail = find_stale_refund_detail(leaky_response, demo_decision)
premature_language = find_premature_approval_language(leaky_response, demo_decision)

print(f"decision.refund_status (ground truth): {demo_decision.refund_status}")
print(f"decision.approved_amount / refund_id  : {demo_decision.approved_amount} / {demo_decision.refund_id}")
print(f"\nComms tried to say            : {leaky_response!r}")
print(f"\nfind_stale_refund_detail       : {stale_detail}")
print(f"find_premature_approval_language: {premature_language}")
print(
    "\nBoth catches fire on a customer_response that cites the wrong amount/id "
    "AND describes an escalated case as already approved -- exactly what "
    "enforce_decision's own callers use to fall back to a safe, generic reply "
    "instead of sending this to the customer."
)


decision.refund_status (ground truth): ESCALATION_REQUIRED
decision.approved_amount / refund_id  : None / None

Comms tried to say            : 'Good news -- your refund of $150.00 has been approved and refund RF-0001 is on its way.'

find_stale_refund_detail       : customer_response cites refund_id 'RF-0001', which does not match decision.refund_id (None) -- likely a stale refund_id repeated from decision.rationale.
find_premature_approval_language: customer_response describes the refund as already approved/processed/issued in 'Good news -- your refund of $150.00 has been approved and refund RF-0001 is on its way.', but decision.refund_status is 'ESCALATION_REQUIRED', not APPROVED -- no money should be described as already handled or imminent.

Both catches fire on a customer_response that cites the wrong amount/id AND describes an escalated case as already approved -- exactly what enforce_decision's own callers use to fall back to a safe, generic reply instead of sending this to the

## Notes

- Full architecture, the flow diagram, authority-separation rationale, and
  the complete crew guardrails table live in [`README.md`](README.md) —
  this notebook only demonstrates the crew running, it doesn't restate the
  design.
- `tests/crew/` (36 tests, no API key needed) covers the same logic paths
  this notebook exercises live, including `test_tool_ownership.py`, which
  asserts each agent's registry only contains the tool names
  `TOOL_OWNERSHIP` assigns it — the authority-separation check above is
  enforced in CI, not just true by construction today.
- A real, saved transcript of `scripts/run_crew_scenarios.py` (stdout +
  structured JSON logs, 6/6 scenarios matched) lives in
  [`docs/evidence/`](docs/evidence/), so the crew's live behavior holds up
  even without your own API key.
- Structured JSON logs from every `handle_ticket()` call above went to
  stderr (not shown in this notebook's cell output), tagged with a
  `case_id` per case; ticket text and customer-facing responses are never
  logged.
- `SLACK_WEBHOOK_URL` is unset in this run, so every alert above was
  written to `starter-kit/outbox/alerts.jsonl` only — no real Slack message
  was sent. Set it (see `.env.example`) to also POST alerts to a real
  incoming webhook.
